<a href="https://colab.research.google.com/github/Annie00000/Project/blob/main/3_30.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 一開始就「點變紅」且「Tabs 有照片」

- **步驟 1：在 Figure 中設定視覺上的「預設變紅」:**
我們使用 selectedpoints 參數。這告訴 Plotly 前端：這張圖一出來，請套用 selected 樣式在這些點上。

In [ ]:
import plotly.graph_objects as go
import pandas as pd

def create_initial_figure(df_focus, default_x, default_y):
    """
    df_focus: 用於畫彩色點的 DataFrame
    default_x, default_y: 想要預設變紅的座標 (x, y)
    """
    fig = go.Figure()

    # ... 畫背景黑點、灰點 (省略) ...

    # --- 繪製重點彩色點 ---
    # 遍歷每一個獨特的 bin 類別來繪製不同的 Trace
    for bin_val in unique_bins:
        df_bin_group = df_focus[df_focus['bin'] == bin_val].reset_index(drop=True)

        # --- 關鍵：尋找預設點在「目前這個 Bin 群組」中的位置 ---
        # 我們在目前的 df_bin_group 中搜尋座標
        matched_idx = df_bin_group[(df_bin_group['x'] == default_x) &
                                  (df_bin_group['y'] == default_y)].index.tolist()
        # 假設 matched_indices 找到了 [5]，代表第 6 筆資料 (0-based)
        # **注意**：這必須是相對於 add_trace 內 x/y 陣列的索引

        fig.add_trace(go.Scatter(
            x=df_bin_group['x'],
            y=df_bin_group['y'],
            mode='markers',
            marker=dict(size=10, line=dict(width=1, color='white')),
            name=f'Bin {bin_val}',
            customdata=df_bin_group['b64_pic'],
            text=df_bin_group['defect_pattern'],

            # --- 這裡安插 selectedpoints ---
            # 如果在這個 Bin 裡面找到了該座標，matched_idx 就會有值 (例如 [0])
            # 如果沒找到，matched_idx 就是 []，此 Trace 就不會有預設紅點
            selectedpoints=matched_idx,

            # 設定選取後的紅色樣式
            selected=dict(marker=dict(color='red', size=13, opacity=1)),
            # 設定未選取時的半透明樣式
            unselected=dict(marker=dict(opacity=0.2)),

            hovertemplate=(
                "Target Bin: %{fullData.name}<br>" +
                "Coord: (%{x}, %{y})<br>" +
                "Pattern: %{text}<extra></extra>"
            )
        ))

    fig.update_layout(clickmode='event+select', ...) # 務必開啟此模式
    return fig

# 在 App 啟動前生成 fig
# fig = create_initial_figure(your_df_focus)

- **步驟 2：Tabs 根據預設位置，產出**

In [ ]:
import dash_bootstrap_components as dbc
from dash import html

def create_photo_tabs(pic_df, die_x, die_y, mode='bin'):
    """
    根據座標與模式動態生成 dbc.Tabs，內容僅包含圖片。
    """
    # 1. 根據指定座標篩選資料
    matched_df = pic_df[(pic_df['Die_X'] == die_x) & (pic_df['Die_Y'] == die_y)].copy()

    if matched_df.empty:
        return html.Div(f"No data for ({die_x}, {die_y})")

    # 2. 建立 Tab 列表
    tab_list = []

    for _, row in matched_df.iterrows():
        # 3. 根據 mode 決定 Tab 標題
        tab_label = str(row['bin']) if mode == 'bin' else str(row['pattern'])

        # 4. 建立 Tab 內容：僅包含 Img
        img_component = html.Img(
            src=f"data:image/png;base64,{row['pic_b64']}",
            style={
                'width': '100%',
                'height': 'auto',
                'display': 'block',
                'marginTop': '10px',
                'objectFit': 'contain', # 確保圖片比例不失真
                'border': '1px solid #ddd',
            }
        )

        # 將圖片包進 Tab
        tab_list.append(
            dbc.Tab(img_component, label=tab_label)
        )

    # 回傳整組 Tabs
    return dbc.Tabs(tab_list)

In [ ]:
# 4. 建立 Tab 內容：僅包含 Img
html.Div([
    html.Img(
        # 核心：展示 Base64 圖片
        src=f"data:image/png;base64,{row['pic_b64']}",
        style={
            'width': '100%',
            'maxHeight': '500px',
            'objectFit': 'contain', # 確保圖片比例不失真
            'border': '1px solid #ddd'
        }
    )
], style={'textAlign': 'center'})

## 3. callback

In [ ]:
import pandas as pd
import dash_bootstrap_components as dbc
from dash import Dash, dcc, html, Input, Output, State, callback, no_update

# --- 1. 準備 pic_df 資料 (示意) ---
# pic_df = pd.read_csv(...)
# 欄位應包含 ['Die_X', 'Die_Y', 'pic_b64', 'pattern', 'bin']

# --- 2. 定義產出 Tabs 的函式 ---
def create_photo_tabs(pic_df, die_x, die_y, mode='bin'):
    # 根據座標篩選資料
    matched_df = pic_df[(pic_df['Die_X'] == die_x) & (pic_df['Die_Y'] == die_y)].copy()

    if matched_df.empty:
        return html.Div(f"No photo data for ({die_x}, {die_y})", style={'padding': '20px'})

    tab_list = []
    for _, row in matched_df.iterrows():
        # 根據 mode 決定 Tab 標題
        tab_label = str(row['bin']) if mode == 'bin' else str(row['pattern'])

        # 內容僅包含 Img
        img_component = html.Img(
            src=f"data:image/png;base64,{row['pic_b64']}",
            style={
                'width': '100%',
                'height': 'auto',
                'display': 'block',
                'marginTop': '10px',
                'border': '1px solid #ddd'
            }
        )
        tab_list.append(dbc.Tab(img_component, label=tab_label))

    return dbc.Tabs(tab_list)

# --- 3. Dash App 佈局 ---
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col([
            # 這裡放你之前繪製好的 Wafer Map (go.Figure)
            dcc.Graph(id='wafer-map', figure=fig)
        ], width=7),

        dbc.Col([
            html.H5("缺陷照片內容"),
            # 切換 Mode 的開關 (供 Callback 使用)
            dbc.RadioItems(
                options=[
                    {"label": "By Bin", "value": "bin"},
                    {"label": "By Type (Pattern)", "value": "type"},
                ],
                value="bin",
                id="mode-selector",
                inline=True,
                style={'marginBottom': '10px'}
            ),
            # **動態 Tabs 的容器**
            html.Div(id='tabs-content-container')
        ], width=5)
    ])
], fluid=True)

# --- 4. Callback 實作 ---
@app.callback(
    Output('tabs-content-container', 'children'),
    Input('wafer-map', 'clickData'),    # 監聽地圖點擊
    Input('mode-selector', 'value'),     # 監聽 Mode 切換 (bin/type)
    prevent_initial_call=False           # 允許初始載入
)
def update_defect_tabs(clickData, current_mode):
    # --- 處理初始狀態 (Default 選取) ---
    # 如果還沒點擊，我們手動指定一個初始座標 (例如 10, 0)
    if clickData is None:
        target_x, target_y = 10, 0
    else:
        # 從 clickData 提取座標
        target_x = clickData['points'][0]['x']
        target_y = clickData['points'][0]['y']

    # --- 呼叫函式生成新的 Tabs ---
    # 這裡會根據最新的座標與模式重新產出整個 dbc.Tabs 元件
    new_tabs = create_photo_tabs(pic_df, target_x, target_y, mode=current_mode)

    return new_tabs

if __name__ == '__main__':
    app.run_server(debug=True)

### 3-1. 點到heatmap的點 → no_update  (只有scatter)

* **方法 A：檢查 fullData.type (最推薦)**

In [ ]:
from dash import no_update

@callback(
    Output('tabs-content-container', 'children'),
    Input('wafer-map', 'clickData')
)
def update_tabs(clickData):
    if clickData is None:
        # 初始狀態：預設顯示某點的照片
        return create_photo_tabs(pic_df, 10, 0, mode='bin')

    # 1. 取得點擊資訊
    point = clickData['points'][0]

    # 2. 判斷點擊的圖層類型
    # Plotly 在 clickData 中會攜帶該點所屬 Trace 的資訊
    trace_type = point['fullData']['type']

    if trace_type == 'heatmap':
        # 如果點到的是底層 Heatmap，直接不更新，維持現狀
        return no_update

    # 3. 既然不是 Heatmap，那就是 Scatter (彩色重點點)
    target_x = point['x']
    target_y = point['y']

    return create_photo_tabs(pic_df, target_x, target_y)

- **方法 B：檢查 curveNumber**

In [ ]:
if point['curveNumber'] == 0:
      return no_update

- **方法 C : 使用 z 排除 Heatmap 的 Callback 範例**

In [ ]:
# --- 核心過濾邏輯：利用 'z' 屬性排除 Heatmap ---
# Heatmap 的點擊資料一定會有 'z'，Scatter 則無
if 'z' in point:
    # 點到的是 Heatmap，不做任何更新
    return no_update

- 點擊 Heatmap 時：

  { "points": [{ "x": 10, "y": 0, "z": 5.2, ... }] }  // <--- 包含 'z'

- 點擊 Scatter 時：

  { "points": [{ "x": 10, "y": 0, "customdata": "...", ... }] } // <--- 不含 'z'